# COM299 Applied Data Science  
## Initial Inspection of the STATS19 Vehicle Dataset

This notebook inspects the 2025 STATS19 vehicle dataset for the SB4 Insurance Analytics Team brief.

The purpose is to understand the structure of the vehicle data, identify vehicle-level factors that may later be associated with casualty severity, and inspect the fields needed to link vehicle records to casualty records.

### Load the vehicle dataset

The first step is to load the raw 2025 STATS19 vehicle CSV into pandas.

At this stage, the dataset is only being read and inspected. No cleaning or transformation is being applied.

In [1]:
import pandas as pd

vehicle_path = "../data/raw/dft-road-casualty-statistics-vehicle-2025.csv"

vehicles = pd.read_csv(vehicle_path)

C:\Users\test\AppData\Local\Temp\ipykernel_4124\2750768681.py:5: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicles = pd.read_csv(vehicle_path)


**Result:** The vehicle dataset loaded successfully, but pandas produced a `DtypeWarning` for the `generic_make_model` column.

This means that column contains mixed data types, such as a combination of text and numeric-style values.

No cleaning decision will be made yet. The warning is recorded so the column can be inspected properly before deciding how it should be handled.

### Check the dataset size

Before inspecting individual vehicle variables, the number of rows and columns should be confirmed.

This provides a basic overview of the scale of the vehicle dataset.

In [2]:
vehicles.shape

(183948, 32)

**Result:** The vehicle dataset contains 183,948 rows and 32 columns.

This means there are substantially more vehicle records than casualty records, which is expected because a collision can involve multiple vehicles and not every vehicle necessarily has a casualty associated with it.

### Inspect the available vehicle columns

The next step is to review the vehicle dataset column names.

This helps identify the fields used to link vehicles to collisions and casualties, as well as the vehicle-level factors that may later be relevant to the SB4 analysis.

In [3]:
vehicles.columns.tolist()

['collision_index',
 'collision_year',
 'collision_ref_no',
 'vehicle_reference',
 'vehicle_type',
 'towing_and_articulation',
 'vehicle_manoeuvre_historic',
 'vehicle_manoeuvre',
 'vehicle_direction_from',
 'vehicle_direction_to',
 'vehicle_location_restricted_lane_historic',
 'vehicle_location_restricted_lane',
 'junction_location',
 'skidding_and_overturning',
 'hit_object_in_carriageway',
 'vehicle_leaving_carriageway',
 'hit_object_off_carriageway',
 'first_point_of_impact',
 'vehicle_left_hand_drive',
 'journey_purpose_of_driver_historic',
 'journey_purpose_of_driver',
 'sex_of_driver',
 'age_of_driver',
 'age_band_of_driver',
 'engine_capacity_cc',
 'propulsion_code',
 'age_of_vehicle',
 'generic_make_model',
 'driver_imd_decile',
 'lsoa_of_driver',
 'escooter_flag',
 'driver_distance_banding']

### Check vehicle-level uniqueness

To understand the grain of the vehicle dataset, it is important to check whether each vehicle is uniquely identified within a collision.

If the combination of `collision_index` and `vehicle_reference` is unique, this supports the idea that one row represents one vehicle within one collision.

In [4]:
vehicles.duplicated(
    subset=["collision_index", "vehicle_reference"]
).sum()

np.int64(0)

**Result:** No duplicate combinations of `collision_index` and `vehicle_reference` were found.

This supports the idea that each row in the vehicle dataset represents one unique vehicle within one collision.

This is important for later joins because it suggests that `collision_index` together with `vehicle_reference` can act as the vehicle-level key when linking vehicle records to casualty records.

### Check missing vehicle linkage fields

Before relying on the vehicle identifiers for later joins, it is important to confirm that the key linkage fields are complete.

Missing values in `collision_index` or `vehicle_reference` could prevent some vehicle records from linking correctly to casualty records.

In [5]:
vehicles[
    ["collision_index", "vehicle_reference"]
].isna().sum()

collision_index      0
vehicle_reference    0
dtype: int64

**Result:** No missing values were found in `collision_index` or `vehicle_reference`.

This means all vehicle records contain the two fields needed for later linkage to casualty records.

Combined with the earlier uniqueness check, this provides strong evidence that `collision_index` plus `vehicle_reference` is a suitable vehicle-level key for validating the casualty-to-vehicle relationship later.

### Check data types for key vehicle fields

Before moving further, the data types of the main identifier and candidate vehicle fields should be checked.

This helps confirm whether the linkage fields are stored consistently and whether any obvious type issues are present before later joins or analysis.

In [6]:
vehicles[
    [
        "collision_index",
        "collision_ref_no",
        "vehicle_reference",
        "vehicle_type",
        "vehicle_manoeuvre",
        "junction_location",
        "age_of_driver",
        "age_of_vehicle"
    ]
].dtypes

collision_index        str
collision_ref_no       str
vehicle_reference    int64
vehicle_type         int64
vehicle_manoeuvre    int64
junction_location    int64
age_of_driver        int64
age_of_vehicle       int64
dtype: object

**Result:** The main vehicle identifiers and selected candidate variables are stored using appropriate data types.

- `collision_index` and `collision_ref_no` are stored as strings, which is appropriate for identifiers.
- `vehicle_reference`, `vehicle_type`, `vehicle_manoeuvre`, `junction_location`, `age_of_driver`, and `age_of_vehicle` are stored as integers.

No immediate data-type changes are required for these fields at this stage.

### Inspect vehicle type categories

The `vehicle_type` field is a coded categorical variable and is likely to be relevant to the SB4 question.

Inspecting its distribution helps identify the available categories, dominant vehicle groups, and any sentinel-style or unusual codes before the field is interpreted.

In [7]:
vehicles["vehicle_type"].value_counts(dropna=False).sort_index()

vehicle_type
-1          9
 1      17021
 2        813
 3       7876
 4       1792
 5       4976
 8       3614
 9     123212
 10       296
 11      3023
 16        78
 17       357
 18        28
 19     12555
 20       884
 21      2633
 22       346
 23       602
 33      1469
 90       770
 97       636
 98       673
 99       285
Name: count, dtype: int64

**Result:** The `vehicle_type` field contains a wide range of coded vehicle categories.

The largest category is code `9`, which represents cars, with 123,212 records. Other substantial groups include pedal cycles, motorcycles, vans/light goods vehicles, and buses/coaches.

Special codes such as `90`, `97`, `98`, `99`, and `-1` require careful handling because they represent other, unknown, or incomplete classifications rather than standard vehicle categories.

This variable is likely to be important for SB4, but the categories may need to be grouped sensibly later to avoid very small groups and to support meaningful statistical comparison.

### Inspect driver age values

The `age_of_driver` field may be relevant to SB4 because driver age could be associated with casualty severity.

Before using the variable, the raw values should be inspected to identify whether sentinel-style values or unusual ages are present.

In [8]:
vehicles["age_of_driver"].value_counts().sort_index().head(20)

age_of_driver
-1     28262
 1         3
 2         2
 3         7
 4        13
 5        20
 6        22
 7        39
 8        72
 9       103
 10      144
 11      265
 12      352
 13      372
 14      418
 15      516
 16      819
 17     2095
 18     3374
 19     3187
Name: count, dtype: int64

**Result:** The `age_of_driver` field contains 28,262 records with a value of `-1`.

A value of `-1` is not a valid driver age, so it should be treated as a coded missing or unknown-style value rather than a real measurement.

This means driver age is not complete for every vehicle record. The effect of these unavailable values will need to be considered before `age_of_driver` is used in later statistical analysis.

### Inspect vehicle age values

The `age_of_vehicle` field may be relevant to SB4 because vehicle age could be associated with casualty severity.

The raw values are inspected first to identify whether missing, sentinel-style, or unusual values are present before the variable is considered for later analysis.

In [9]:
vehicles["age_of_vehicle"].value_counts().sort_index().head(20)

age_of_vehicle
-1    183948
Name: count, dtype: int64

**Result:** Every record in `age_of_vehicle` is coded as `-1`.

This means the field does not contain usable vehicle-age information for the 2025 vehicle dataset.

As a result, `age_of_vehicle` should not be considered a suitable SB4 explanatory variable unless the official documentation provides another interpretation or an alternative source for vehicle age.

### Inspect driver sex categories

The `sex_of_driver` field may be relevant to SB4 as a driver-level characteristic.

Before using it, the coded values should be inspected to identify the main categories and any unknown or sentinel-style values.

In [10]:
vehicles["sex_of_driver"].value_counts(dropna=False).sort_index()

sex_of_driver
-1      4307
 1    112413
 2     47045
 3     20183
Name: count, dtype: int64

### Inspect engine capacity values

The `engine_capacity_cc` field may be relevant to SB4 as a vehicle characteristic.

Before using it, the raw values should be inspected to identify whether sentinel-style values or unrealistic entries are present.

In [11]:
vehicles["engine_capacity_cc"].value_counts().sort_index().head(20)

engine_capacity_cc
-1      44869
 1          1
 8          1
 12         1
 48        12
 49       264
 50       108
 75         1
 79         1
 85         3
 89         1
 96         1
 97         1
 98         3
 99         1
 100        4
 108      236
 109       24
 110      238
 113       29
Name: count, dtype: int64

**Result:** The `engine_capacity_cc` field contains 44,869 records coded as `-1`.

This indicates that engine capacity is unavailable or unknown for a substantial proportion of vehicle records.

The remaining values also include some very small engine capacities, so the field will need further validation against vehicle type and the official documentation before it is used in analysis.

`engine_capacity_cc` may still be useful for SB4, but its missingness and suitability will need to be assessed before it is selected as a final explanatory variable.

### Inspect vehicle propulsion categories

The `propulsion_code` field describes the vehicle's propulsion or fuel type and may be relevant to the SB4 analysis.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values before interpretation.

In [12]:
vehicles["propulsion_code"].value_counts(dropna=False).sort_index()

propulsion_code
-1     38869
 1     74024
 2     50184
 3      6060
 5        64
 6         5
 7        74
 8     14193
 9         1
 10        1
 12      473
Name: count, dtype: int64

**Result:** The `propulsion_code` field contains several coded categories, with `-1` appearing in 38,869 records.

This indicates that propulsion information is unavailable or unknown for a substantial proportion of vehicles.

The remaining numeric values represent specific propulsion types, but they should not be interpreted until the official STATS19/STATS20 lookup is consulted.

This variable may still be useful for SB4, but its missingness and category distribution will need to be considered before it is selected for the final analysis.

### Inspect junction location categories

The `junction_location` field describes where the vehicle was in relation to a junction.

This may be relevant to SB4 because junction context could be associated with casualty severity.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [13]:
vehicles["junction_location"].value_counts(dropna=False).sort_index()

junction_location
-1     2456
 0    76273
 1    38649
 2     9553
 3     2971
 4     5494
 5     4706
 6     8134
 7      417
 8    26845
 9     8450
Name: count, dtype: int64

**Result:** The `junction_location` field contains several coded categories, with `-1` appearing in 2,456 records.

The majority of records fall within the standard coded categories, so this variable appears to have much better coverage than fields such as `age_of_vehicle`, `engine_capacity_cc`, and `propulsion_code`.

The exact meaning of each code still needs to be confirmed using the official STATS19/STATS20 lookup before interpretation.

Because junction context is a circumstantial factor and the field has relatively good coverage, `junction_location` may be a strong candidate for later SB4 analysis.

### Inspect vehicle manoeuvre categories

The `vehicle_manoeuvre` field describes what the vehicle was doing at the time of the collision.

This may be relevant to SB4 because different manoeuvres could be associated with different casualty severity outcomes.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [15]:
vehicles["vehicle_manoeuvre"].value_counts(dropna=False).sort_index()

vehicle_manoeuvre
-1      5198
 1      2217
 2      8143
 3      7295
 4      9590
 5      8818
 6      1207
 7      6038
 8       856
 9     16304
 10     2199
 11     1406
 12     1613
 13     3400
 14     1536
 15     1257
 19    93399
 20      389
 99    13083
Name: count, dtype: int64

**Result:** The `vehicle_manoeuvre` field contains a wide range of coded categories.

The most frequent value is code `19`, with 93,399 records, while code `99` also appears frequently with 13,083 records. The value `-1` appears in 5,198 records.

The exact meaning of these codes must be confirmed using the official STATS19/STATS20 lookup before interpretation.

Because this variable has many categories, any later use in the SB4 analysis may require sensible grouping to avoid very small categories and make comparisons easier to interpret.

### Inspect skidding and overturning categories

The `skidding_and_overturning` field records whether the vehicle experienced skidding or overturning during the collision.

This may be relevant to SB4 because these events could be associated with more severe casualty outcomes.

The coded values are inspected first to identify the available categories and any unknown or sentinel-style values.

In [16]:
vehicles["skidding_and_overturning"].value_counts(dropna=False).sort_index()

skidding_and_overturning
-1      5435
 0    145130
 1     10606
 2      3293
 3        41
 4        22
 5      5286
 9     14135
Name: count, dtype: int64

**Result:** The `skidding_and_overturning` field contains several coded categories.

Most records are coded as `0`, with smaller groups across the remaining categories. The values `-1` and `9` also appear and should be checked against the official STATS19/STATS20 lookup before interpretation.

Because the variable has relatively few categories and may reflect more severe collision circumstances, it could be a useful candidate for later SB4 analysis.

### Check missing values across the vehicle dataset

Missing data can affect whether individual vehicle variables are suitable for later analysis.

A first-pass missing-value check is used to identify any standard `NaN` values across the vehicle dataset before cleaning decisions are made.

In [17]:
vehicles.isna().sum()

collision_index                              0
collision_year                               0
collision_ref_no                             0
vehicle_reference                            0
vehicle_type                                 0
towing_and_articulation                      0
vehicle_manoeuvre_historic                   0
vehicle_manoeuvre                            0
vehicle_direction_from                       0
vehicle_direction_to                         0
vehicle_location_restricted_lane_historic    0
vehicle_location_restricted_lane             0
junction_location                            0
skidding_and_overturning                     0
hit_object_in_carriageway                    0
vehicle_leaving_carriageway                  0
hit_object_off_carriageway                   0
first_point_of_impact                        0
vehicle_left_hand_drive                      0
journey_purpose_of_driver_historic           0
journey_purpose_of_driver                    0
sex_of_driver

**Result:** No standard missing values (`NaN`) were found in any of the vehicle dataset columns.

This suggests the vehicle dataset is structurally complete in terms of blank or null values.

However, several fields already contain sentinel-style values such as `-1` and other coded categories that represent unavailable, unknown, or special cases. Therefore, the absence of `NaN` values does not mean the dataset is fully complete or analysis-ready.

These coded values will need to be identified and handled using the official STATS19/STATS20 definitions before cleaning or statistical analysis.

### Inspect `generic_make_model`

The `generic_make_model` column triggered a pandas `DtypeWarning` when the vehicle dataset was loaded.

This suggests that the column contains mixed data types, so a small sample and its Python data type should be inspected before deciding whether any conversion is needed.

In [18]:
vehicles["generic_make_model"].head(20)

0     MAKE AND MODEL REDACTED
1     MAKE AND MODEL REDACTED
2     MAKE AND MODEL REDACTED
3     MAKE AND MODEL REDACTED
4                 AUDI E-TRON
5                 AUDI E-TRON
6                 AUDI E-TRON
7                 AUDI E-TRON
8                 AUDI E-TRON
9                 AUDI E-TRON
10                AUDI E-TRON
11                AUDI E-TRON
12                AUDI E-TRON
13                AUDI E-TRON
14                AUDI E-TRON
15                AUDI E-TRON
16                AUDI E-TRON
17                AUDI E-TRON
18                AUDI E-TRON
19                AUDI E-TRON
Name: generic_make_model, dtype: object

In [19]:
vehicles["generic_make_model"].map(type).value_counts()

generic_make_model
<class 'str'>    151180
<class 'int'>     32768
Name: count, dtype: int64

**Result:** The `generic_make_model` column contains mixed Python data types:

- 151,180 values are stored as strings
- 32,768 values are stored as integers

This explains the pandas `DtypeWarning` seen when the vehicle dataset was loaded.

Because the column represents make/model information, it should be treated as a categorical/text field rather than as a numeric measurement. No conversion is being applied yet; the mixed types are simply recorded as a data-quality issue for later cleaning.**Result:** The `generic_make_model` column contains mixed Python data types:

- 151,180 values are stored as strings
- 32,768 values are stored as integers

This explains the pandas `DtypeWarning` seen when the vehicle dataset was loaded.

Because the column represents make/model information, it should be treated as a categorical/text field rather than as a numeric measurement. No conversion is being applied yet; the mixed types are simply recorded as a data-quality issue for later cleaning.
